# BoFM EModE parser — train on PCEEC (UD), parse the Book of Mormon

Trains a biaffine dependency parser with a **historical-English** encoder (MacBERTh, BERT
pretrained on English 1450–1950) on the 2.3M-token PCEEC→UD data, then re-parses the Book of
Mormon. Output `bofm_parsed.conllu` → send back to Claude to build Text-Fabric **v0.2** (replaces
the weak Stanza modern-English syntax layer).

**Runtime → Change runtime type → GPU (T4 is fine).** Then run cells top to bottom.
Upload `train-package.zip` (Claude prepared it) when the upload cell prompts.

In [ ]:
# 1. GPU check + install
!nvidia-smi -L
!pip install -q supar==1.1.4 'transformers>=4.20,<4.40'
import supar, torch; print('supar', supar.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2. Upload train-package.zip (contains train/dev/test.conllu + bofm_toparse.conllu)
from google.colab import files
import zipfile, os
up = files.upload()                      # pick train-package.zip
zname = next(iter(up))
with zipfile.ZipFile(zname) as z: z.extractall('data')
# flatten if zipped with a folder
for root,_,fs in os.walk('data'):
    for f in fs:
        if f.endswith('.conllu') and not os.path.exists(f): os.rename(os.path.join(root,f), f)
print(sorted(f for f in os.listdir('.') if f.endswith('.conllu')))

In [ ]:
# 3. Train the biaffine dependency parser with the MacBERTh historical-English encoder
#    (~30-60 min on a T4 for 87k sentences; watch the dev LAS climb)
!python -m supar.cmds.biaffine_dep train -b -d 0     -p bofm-emode.parser -f bert --bert emanjavacas/MacBERTh     --train train.conllu --dev dev.conllu --test test.conllu     --batch-size 1000 --epochs 10

In [ ]:
# 4. Evaluate on held-out test (LAS/UAS — how well it learned EModE attachment)
!python -m supar.cmds.biaffine_dep evaluate -d 0 -p bofm-emode.parser --data test.conllu

In [ ]:
# 5. Re-parse the Book of Mormon (predict heads/deprels on the existing tokenization)
!python -m supar.cmds.biaffine_dep predict -d 0 -p bofm-emode.parser     --data bofm_toparse.conllu --pred bofm_parsed.conllu
# sanity peek
print(open('bofm_parsed.conllu',encoding='utf-8').read()[:1200])

In [ ]:
# 6. Download the re-parse (send this back to Claude for TF v0.2) + the trained parser
from google.colab import files
files.download('bofm_parsed.conllu')
!zip -qr bofm-emode-parser.zip bofm-emode.parser && echo 'parser zipped'
files.download('bofm-emode-parser.zip')

## Back on the desktop
- `bofm_parsed.conllu` → Claude maps it onto the TF tokens, rewrites `deprel`/`head` → **TF v0.2**.
- Then the binding rules run on the *good* EModE syntax → the bulk over-split classes
  (rule_19 ~959, rule_07 ~599, polysyndeton ~350).
- `bofm-emode-parser.zip` = the trained parser (re-runnable locally if torch is fixed, or kept for re-parses).